In [11]:
# =========================================================
# STEP 1: Data Loading, Encoding, and Splitting (30-Day)
# =========================================================
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Loading 30-day model ready data...")
data_path = '../data/final_model_ready_pune_data_30day.csv'
df = pd.read_csv(data_path)
df['arrival_date'] = pd.to_datetime(df['arrival_date'])

# 1. Encode Categoricals for LightGBM
categorical_cols = ['mandi_name', 'district', 'state', 'variety']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

# 2. Chronological Train/Test Split
# We keep Nov 1st, 2024 as the split to maintain strict consistency across all 3 horizons
split_date = '2024-11-01' 
train_df = df[df['arrival_date'] < split_date].copy()
test_df = df[df['arrival_date'] >= split_date].copy()

# 3. Drop Leaky Columns
drop_cols = [
    'arrival_date', 'target_price', 'commodity', 
    'modal_price', 'min_price', 'max_price'
]

X_train = train_df.drop(columns=drop_cols, errors='ignore')
y_train = train_df['target_price']

X_test = test_df.drop(columns=drop_cols, errors='ignore')
y_test = test_df['target_price']

print("\n--- CHECKPOINT 1 COMPLETE ---")
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape:  {X_test.shape}")
print(f"Features used: {list(X_train.columns)}")

Loading 30-day model ready data...

--- CHECKPOINT 1 COMPLETE ---
Training features shape: (17660, 27)
Testing features shape:  (6619, 27)
Features used: ['mandi_name', 'district', 'state', 'variety', 'is_real_trade', 'day_of_week', 'month', 'day_of_year', 'is_weekend', 'is_holiday', 'price_lag_30', 'price_lag_31', 'price_lag_32', 'price_lag_45', 'price_roll_mean_7', 'price_roll_std_7', 'price_roll_mean_30', 'price_expanding_mean', 'sin_365_1', 'cos_365_1', 'sin_365_2', 'cos_365_2', 'temp_mean_lag30', 'rainfall_lag30', 'rainfall_7d_sum', 'rainfall_30d_sum', 'temp_7d_avg']


In [2]:
# =========================================================
# STEP 2: LightGBM Datasets & Tuned 30-Day Training
# =========================================================
import lightgbm as lgb
import os

print("Creating LightGBM datasets for 30-Day Horizon...")

# 1. Create Base Datasets
lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_cols, free_raw_data=False)
lgb_eval = lgb.Dataset(X_test, label=y_test, categorical_feature=categorical_cols, reference=lgb_train, free_raw_data=False)

# 2. Define Base Parameters (Slow learning rate for stability over long horizons)
base_params = {
    'boosting_type': 'gbdt',
    'learning_rate': 0.01, 
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42
}

# 3. Quantile-Specific Tuning (Wide alphas 0.05 / 0.95 to capture high monthly volatility)
quantile_configs = {
    'p50': {
        'alpha': 0.50, 'max_depth': 10, 'num_leaves': 63, 
        'min_data_in_leaf': 15, 'lambda_l1': 0.1, 'lambda_l2': 0.1
    },
    'p10': {
        'alpha': 0.05, 'max_depth': 6, 'num_leaves': 31, 
        'min_data_in_leaf': 30, 'lambda_l1': 1.0, 'lambda_l2': 1.0
    },
    'p90': {
        'alpha': 0.95, 'max_depth': 6, 'num_leaves': 31, 
        'min_data_in_leaf': 30, 'lambda_l1': 1.0, 'lambda_l2': 1.0
    }
}

models = {}
os.makedirs('../models', exist_ok=True)
print("\nStarting 30-Day Horizon Probabilistic Training...")

# 4. Train the 3 Models
for name, config in quantile_configs.items():
    print(f"\n--- Training {name} Model (Alpha={config['alpha']}) ---")
    
    # Merge base params with specific config
    params = {**base_params, **config}
    params['objective'] = 'quantile'
    params['metric'] = 'quantile'

    callbacks = [
        lgb.early_stopping(stopping_rounds=100, first_metric_only=False),
        lgb.log_evaluation(period=200)
    ]

    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=3000, 
        valid_sets=[lgb_train, lgb_eval],
        valid_names=['train', 'eval'],
        callbacks=callbacks
    )
    
    models[name] = model
    
    # Save the model
    model_path = f"../models/lightgbm_onion_30day_{name}.txt"
    model.save_model(model_path)
    print(f"Saved: {model_path}")

print("\n--- CHECKPOINT 2 COMPLETE: All 30-Day Models Trained & Saved ---")

Creating LightGBM datasets for 30-Day Horizon...

Starting 30-Day Horizon Probabilistic Training...

--- Training p50 Model (Alpha=0.5) ---
Training until validation scores don't improve for 100 rounds
[200]	train's quantile: 101.644	eval's quantile: 262.701
Early stopping, best iteration is:
[196]	train's quantile: 102.702	eval's quantile: 262.566
Saved: ../models/lightgbm_onion_30day_p50.txt

--- Training p10 Model (Alpha=0.05) ---
Training until validation scores don't improve for 100 rounds
[200]	train's quantile: 35.5594	eval's quantile: 44.148
Early stopping, best iteration is:
[125]	train's quantile: 38.2642	eval's quantile: 43.1394
Saved: ../models/lightgbm_onion_30day_p10.txt

--- Training p90 Model (Alpha=0.95) ---
Training until validation scores don't improve for 100 rounds
[200]	train's quantile: 42.0981	eval's quantile: 150.108
Early stopping, best iteration is:
[122]	train's quantile: 52.1362	eval's quantile: 142.389
Saved: ../models/lightgbm_onion_30day_p90.txt

--- CHE

In [3]:
# =========================================================
# STEP 3: Generate Formal Performance Metrics (30-Day)
# =========================================================
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

print("Generating 30-Day predictions on unseen test data...\n")

# Create a results dataframe to hold true values and predictions
results_df = test_df[['arrival_date', 'mandi_name', 'target_price']].copy()

# Generate predictions for all 3 quantiles
for name in ['p10', 'p50', 'p90']:
    results_df[f'{name}_pred'] = models[name].predict(X_test)

# 1. Standard Regression Metrics (Using p50 Median Forecast)
mae = mean_absolute_error(results_df['target_price'], results_df['p50_pred'])
rmse = np.sqrt(mean_squared_error(results_df['target_price'], results_df['p50_pred']))
mape = mean_absolute_percentage_error(results_df['target_price'], results_df['p50_pred'])

# 2. Probabilistic Metrics (Using p10 and p90 Bounds)
results_df['in_bound'] = (results_df['target_price'] >= results_df['p10_pred']) & (results_df['target_price'] <= results_df['p90_pred'])
coverage = results_df['in_bound'].mean() * 100

print("="*50)
print("FINAL 30-DAY HORIZON METRICS FOR YOUR PAPER")
print("="*50)
print(f"MAE:  ₹ {mae:.2f}")
print(f"RMSE: ₹ {rmse:.2f}")
print(f"MAPE: {mape*100:.2f}%")
print(f"80% Prediction Interval Coverage: {coverage:.1f}%")
print("="*50)

Generating 30-Day predictions on unseen test data...

FINAL 30-DAY HORIZON METRICS FOR YOUR PAPER
MAE:  ₹ 525.13
RMSE: ₹ 824.41
MAPE: 30.05%
80% Prediction Interval Coverage: 84.4%


In [12]:
# =========================================================
# REVISED 30-DAY MODEL: SMOOTHED TARGET (7-DAY AVERAGE)
# =========================================================
import lightgbm as lgb
import os
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

print("Calculating 7-Day Smoothed Target...")

# 1. Create the smoothed target (7-day average, centered)
train_df['target_price_smooth'] = train_df.groupby('mandi_name')['target_price'].transform(lambda x: x.rolling(window=7, center=True, min_periods=1).mean())
test_df['target_price_smooth'] = test_df.groupby('mandi_name')['target_price'].transform(lambda x: x.rolling(window=7, center=True, min_periods=1).mean())

y_train_smooth = train_df['target_price_smooth']
y_test_smooth = test_df['target_price_smooth']

# 2. Create Datasets with SMOOTHED targets
lgb_train_smooth = lgb.Dataset(X_train, label=y_train_smooth, categorical_feature=categorical_cols, free_raw_data=False)
lgb_eval_smooth = lgb.Dataset(X_test, label=y_test_smooth, categorical_feature=categorical_cols, reference=lgb_train_smooth, free_raw_data=False)

base_params = {
    'boosting_type': 'gbdt',
    'learning_rate': 0.01, 
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42
}

# 3. Standard Quantile Tuning
quantile_configs = {
    'p50': {'alpha': 0.50, 'max_depth': 10, 'num_leaves': 63, 'min_data_in_leaf': 15, 'lambda_l1': 0.1, 'lambda_l2': 0.1},
    'p10': {'alpha': 0.05, 'max_depth': 6, 'num_leaves': 31, 'min_data_in_leaf': 30, 'lambda_l1': 1.0, 'lambda_l2': 1.0},
    'p90': {'alpha': 0.95, 'max_depth': 6, 'num_leaves': 31, 'min_data_in_leaf': 30, 'lambda_l1': 1.0, 'lambda_l2': 1.0}
}

models_smooth = {}
print("\nTraining Smoothed 30-Day Models...")

for name, config in quantile_configs.items():
    params = {**base_params, **config}
    params['objective'] = 'quantile'
    params['metric'] = 'quantile'

    callbacks = [lgb.early_stopping(stopping_rounds=100, first_metric_only=False), lgb.log_evaluation(period=0)]
    
    model = lgb.train(
        params, lgb_train_smooth, num_boost_round=3000, 
        valid_sets=[lgb_train_smooth, lgb_eval_smooth], valid_names=['train', 'eval'],
        callbacks=callbacks
    )
    models_smooth[name] = model
    
    # Save the model
    model_path = f"../models/lightgbm_onion_30day_{name}_smoothed.txt"
    model.save_model(model_path)
    print(f"Saved: {model_path}")

print("\nGenerating predictions on test data...")

results_df = test_df[['arrival_date', 'mandi_name', 'target_price_smooth']].copy()

# 4. Predict
for name in ['p10', 'p50', 'p90']:
    results_df[f'{name}_pred'] = models_smooth[name].predict(X_test)

# 5. Calculate Final Metrics against SMOOTHED prices
mae = mean_absolute_error(results_df['target_price_smooth'], results_df['p50_pred'])
rmse = np.sqrt(mean_squared_error(results_df['target_price_smooth'], results_df['p50_pred']))
mape = mean_absolute_percentage_error(results_df['target_price_smooth'], results_df['p50_pred'])

results_df['in_bound'] = (results_df['target_price_smooth'] >= results_df['p10_pred']) & (results_df['target_price_smooth'] <= results_df['p90_pred'])
coverage = results_df['in_bound'].mean() * 100

print("="*50)
print("SMOOTHED TARGET 30-DAY HORIZON METRICS")
print("="*50)
print(f"MAE:  ₹ {mae:.2f}")
print(f"RMSE: ₹ {rmse:.2f}")
print(f"MAPE: {mape*100:.2f}%")
print(f"80% Prediction Interval Coverage: {coverage:.1f}%")
print("="*50)


Calculating 7-Day Smoothed Target...

Training Smoothed 30-Day Models...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[376]	train's quantile: 61.6171	eval's quantile: 255.459
Saved: ../models/lightgbm_onion_30day_p50_smoothed.txt
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[138]	train's quantile: 36.1809	eval's quantile: 41.4166
Saved: ../models/lightgbm_onion_30day_p10_smoothed.txt
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[11]	train's quantile: 83.8971	eval's quantile: 152.908
Saved: ../models/lightgbm_onion_30day_p90_smoothed.txt

Generating predictions on test data...
SMOOTHED TARGET 30-DAY HORIZON METRICS
MAE:  ₹ 510.92
RMSE: ₹ 812.06
MAPE: 28.72%
80% Prediction Interval Coverage: 87.7%
